In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [ ]:
import numpy as np, pandas as pd, torch
from dataclasses import dataclass
from typing import Any
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from transformers import (AutoFeatureExtractor, AutoModelForAudioClassification,
                          TrainingArguments, Trainer)

In [ ]:
CHECKPOINT = "facebook/wav2vec2-base"
MAX_LEN = 6 * 16000
PER_GROUP = 100
SEED = 42

### 2. Φόρτωση του ESD_english και δημιουργία υποσυνόλου

In [ ]:
dataset = load_dataset("AbstractTTS/ESD_english", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

speakers = [f.split("_")[0] for f in dataset["file"]]
df = pd.DataFrame({"speaker": speakers, "emotion": dataset["emotion"]})
print(df["speaker"].nunique(), "ομιλητές |", sorted(df["emotion"].unique()))

rng = np.random.RandomState(SEED)
keep = []
for (spk, emo), g in df.groupby(["speaker", "emotion"]):
    idx = g.index.to_numpy()
    keep.extend(rng.choice(idx, size=min(PER_GROUP, len(idx)), replace=False))
dataset = dataset.select(sorted(int(i) for i in keep))
print("Μέγεθος υποσυνόλου:", len(dataset))

### 3. Split ανεξάρτητο ως προς τον ομιλητή

In [ ]:
def add_speaker(batch):
    batch["speaker_id"] = batch["file"].split("_")[0]
    return batch

dataset = dataset.map(add_speaker, load_from_cache_file=False)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))
dataset = DatasetDict({"train": dataset.select(train_idx), "test": dataset.select(test_idx)})

train_spk, test_spk = set(dataset["train"]["speaker_id"]), set(dataset["test"]["speaker_id"])
assert not (train_spk & test_spk), "Επικάλυψη ομιλητών!"
print("Train ομιλητές:", sorted(train_spk))
print("Test ομιλητές:", sorted(test_spk))
print("Φύλο test ομιλητών:", sorted(set(zip(dataset["test"]["speaker_id"], dataset["test"]["gender"]))))
print(len(dataset["train"]), "train |", len(dataset["test"]), "test")

### 4. Ετικέτες

In [ ]:
labels = sorted(set(dataset["train"]["emotion"]))
num_labels = len(labels)
label2id = {n: i for i, n in enumerate(labels)}
id2label = {i: n for n, i in label2id.items()}
print(label2id)

### 5. Προεπεξεργασία (feature extractor)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT)

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        max_length=MAX_LEN, truncation=True
    ).input_values[0]
    batch["labels"] = label2id[batch["emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset,
                              remove_columns=dataset["train"].column_names,
                              load_from_cache_file=False)

### 6. Μοντέλο

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    CHECKPOINT, num_labels=num_labels, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
print(type(model).__name__, [n for n, _ in model.named_children()])

### 7. Collator και μετρικές

In [ ]:
@dataclass
class DataCollatorWithPadding:
    feature_extractor: Any
    def __call__(self, features):
        batch = self.feature_extractor.pad(
            [{"input_values": f["input_values"]} for f in features],
            padding=True, return_tensors="pt",
        )
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=-1)
    all_confusion_matrices.append(confusion_matrix(y, preds, labels=list(range(num_labels))))
    return {
        "accuracy": accuracy_score(y, preds),
        "balanced_accuracy": balanced_accuracy_score(y, preds),
        "macro_f1": f1_score(y, preds, average="macro"),
    }

### 8. Εκπαίδευση

In [ ]:
import math

BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 8

steps_per_epoch = math.ceil(len(encoded_dataset["train"]) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(0.1 * total_steps)          # 10% των συνολικών βημάτων
print(f"steps/epoch={steps_per_epoch} | total={total_steps} | warmup={warmup_steps}")

training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned-ESD",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    fp16=False,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    save_only_model=True,
    seed=SEED,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=feature_extractor,
    data_collator=DataCollatorWithPadding(feature_extractor),
    compute_metrics=compute_metrics,
)
trainer.train()

### 9. Τελική αξιολόγηση του καλύτερου μοντέλου

In [ ]:
from scipy.special import softmax
from sklearn.metrics import (precision_score, recall_score, matthews_corrcoef,
                             cohen_kappa_score, log_loss, roc_auc_score,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

logits = pred.predictions
probs = softmax(logits, axis=-1)
y_bin = label_binarize(y_true, classes=list(range(num_labels)))

full_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "mcc": matthews_corrcoef(y_true, y_pred),
    "kappa": cohen_kappa_score(y_true, y_pred),
    "log_loss": log_loss(y_true, probs, labels=list(range(num_labels))),
    "roc_auc_macro_ovr": roc_auc_score(y_bin, probs, average="macro", multi_class="ovr"),
    "pr_auc_macro": average_precision_score(y_bin, probs, average="macro"),
}
for k, v in full_metrics.items():
    print(f"{k:22s} {v:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιουργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δημιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### 10. Αποθήκευση στο Hugging Face Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/wav2vec2-finetuned-ESD"

model.push_to_hub(repo_id)
feature_extractor.push_to_hub(repo_id)
